In [0]:
dbutils.widgets.text("catalog", "fk_ce_2026_07_1363")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Create target tables (parameterized with catalog)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.orders (
        order_id STRING,
        customer_id STRING,
        store_id STRING,
        order_ts TIMESTAMP,
        order_date DATE,
        order_status STRING,
        payment_method STRING,
        channel STRING,
        promised_mins INT,
        actual_mins INT,
        delivery_fee DECIMAL(10,2),
        updated_at TIMESTAMP
    ) USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.order_items (
        order_id STRING,
        line_number INT,
        product_id STRING,
        quantity INT,
        unit_price DECIMAL(10,2),
        discount_amount DECIMAL(10,2),
        gross_amount DECIMAL(12,2),
        net_amount DECIMAL(12,2),
        updated_at TIMESTAMP
    ) USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.quarantine_orders (
        order_id STRING,
        reject_reason STRING,
        updated_at TIMESTAMP,
        raw_record STRING,
        quarantined_at TIMESTAMP
    ) USING DELTA
""")

# 2. Read bronze orders
df_raw = spark.read.table(f"{catalog}.bronze.orders")

# Bronze stores nested JSON as strings; parse into proper struct/array types
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType

payment_schema = StructType([
    StructField("method", StringType()),
    StructField("txn_ref", StringType())
])
delivery_schema = StructType([
    StructField("promised_mins", IntegerType()),
    StructField("actual_mins", IntegerType()),
    StructField("delivery_fee", DoubleType())
])
item_schema = StructType([
    StructField("product_id", StringType()),
    StructField("qty", IntegerType()),
    StructField("unit_price", DoubleType()),
    StructField("discount", DoubleType())
])
items_schema = ArrayType(item_schema)

bronze_columns = df_raw.columns
df_raw = df_raw.withColumn("raw_record", F.to_json(F.struct([F.col(c) for c in bronze_columns])))
df_raw = (df_raw
    .withColumn("payment", F.from_json(F.col("payment"), payment_schema))
    .withColumn("delivery", F.from_json(F.col("delivery"), delivery_schema))
    .withColumn("items", F.from_json(F.col("items"), items_schema))
)

# 3. C1 - Standardisation
df_std = df_raw.select(
        F.col("raw_record"),
        F.when(F.trim(F.col("order_id")) == "", None).otherwise(F.upper(F.trim(F.col("order_id")))).alias("order_id"),
        F.when(F.trim(F.col("customer_id")) == "", None).otherwise(F.upper(F.trim(F.col("customer_id")))).alias("customer_id"),
        F.when(F.trim(F.col("store_id")) == "", None).otherwise(F.upper(F.trim(F.col("store_id")))).alias("store_id"),

        # S2: order_status standardization
        F.when(F.upper(F.trim(F.col("order_status"))) == "CANCELED", "CANCELLED")
         .otherwise(F.upper(F.trim(F.col("order_status")))).alias("order_status"),

        # S3: payment.method (struct field after re-ingestion)
        F.upper(F.trim(F.col("payment.method"))).alias("payment_method"),

        # S4: channel (not in source data, default to UNKNOWN)
        F.lit("UNKNOWN").alias("channel"),

        # S5: order_ts parsing (handle both 'T' and space separators)
        F.coalesce(
            F.try_to_timestamp(F.col("order_ts"), F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp(F.col("order_ts"), F.lit("yyyy-MM-dd'T'HH:mm:ss"))
        ).alias("order_ts"),

        # S6: updated_at parsing
        F.try_to_timestamp(F.col("updated_at"), F.lit("yyyy-MM-dd HH:mm:ss")).alias("updated_at"),

        # Delivery struct fields
        F.col("delivery.promised_mins").cast("int").alias("promised_mins"),
        F.col("delivery.actual_mins").cast("int").alias("actual_mins"),
        F.col("delivery.delivery_fee").cast("decimal(10,2)").alias("delivery_fee"),
        F.col("items")
    )

df_std = df_std.withColumn("order_date", F.to_date(F.col("order_ts")))

# 4. C2 - De-duplication: keep latest updated_at per order_id
window_spec = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc(), F.col("order_ts").desc())
df_dedup = df_std.withColumn("rn", F.row_number().over(window_spec)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_dedup.createOrReplaceTempView("v_dedup_orders")

# 5. Load reference data for validation
active_customers = spark.table(f"{catalog}.silver.customers_type2").filter(F.col("is_current") == True).select("customer_id").distinct()
active_stores = spark.table(f"{catalog}.silver.stores").select("store_id").distinct()
active_products = spark.table(f"{catalog}.silver.products_type1").select("product_id").distinct()

active_customers.createOrReplaceTempView("ref_customers")
active_stores.createOrReplaceTempView("ref_stores")
active_products.createOrReplaceTempView("ref_products")
df_dedup.createOrReplaceTempView("incoming_orders")

# 6. C3 - Validation & Quarantine (R1-R5)
spark.sql("""
    CREATE OR REPLACE TEMP VIEW v_validated_orders AS
    SELECT 
        o.*,
        CASE 
            WHEN o.order_id IS NULL THEN 'INVALID_ORDER_ID'
            WHEN o.customer_id IS NULL THEN 'MISSING_CUSTOMER'
            WHEN c.customer_id IS NULL THEN 'UNKNOWN_CUSTOMER'
            WHEN o.store_id IS NULL OR s.store_id IS NULL THEN 'UNKNOWN_STORE'
            WHEN o.order_ts IS NULL THEN 'INVALID_TIMESTAMP'
            WHEN o.items IS NULL OR size(o.items) = 0 THEN 'EMPTY_ORDER'
            WHEN o.order_status NOT IN ('PLACED', 'DELIVERED', 'CANCELLED') THEN 'INVALID_STATUS'
            ELSE NULL
        END AS quarantine_reason
    FROM incoming_orders o
    LEFT JOIN ref_customers c ON o.customer_id = c.customer_id
    LEFT JOIN ref_stores s ON o.store_id = s.store_id
""")

df_valid = spark.sql("SELECT * FROM v_validated_orders WHERE quarantine_reason IS NULL")
df_invalid = spark.sql("SELECT * FROM v_validated_orders WHERE quarantine_reason IS NOT NULL")

# 7. Item-level validations (R6-R8) with posexplode for line_number
df_valid_exploded = df_valid.select(
    "order_id", "customer_id", "store_id", "order_ts", "order_date", 
    "order_status", "payment_method", "channel", "promised_mins", 
    "actual_mins", "delivery_fee", "updated_at", "raw_record",
    F.posexplode("items").alias("line_idx", "item")
).select(
    "*",
    (F.col("line_idx") + 1).alias("line_number"),
    F.upper(F.trim(F.col("item.product_id"))).alias("product_id"),
    F.expr("try_cast(item.qty as int)").alias("quantity"),
    F.expr("try_cast(item.unit_price as decimal(10,2))").alias("unit_price"),
    F.coalesce(F.expr("try_cast(item.discount as decimal(10,2))"), F.lit(0.00)).alias("discount_amount")
)

df_valid_items_checked = df_valid_exploded.join(
    active_products.withColumnRenamed("product_id", "p_id"),
    df_valid_exploded.product_id == F.col("p_id"),
    "left"
).withColumn(
    "item_reject_reason",
    F.when(F.col("product_id").isNull() | F.col("p_id").isNull(), "UNKNOWN_PRODUCT")
     .when(F.col("quantity").isNull() | (F.col("quantity") <= 0), "INVALID_QUANTITY")
     .when(F.col("unit_price").isNull() | (F.col("unit_price") <= 0), "INVALID_PRICE")
     .otherwise(None)
)

df_bad_items = df_valid_items_checked.filter(F.col("item_reject_reason").isNotNull()) \
    .groupBy("order_id", "updated_at", "raw_record") \
    .agg(F.min("item_reject_reason").alias("reject_reason"))

# Calculate gross and net amounts for valid items
df_good_items = df_valid_items_checked.filter(F.col("item_reject_reason").isNull()) \
    .withColumn("gross_amount", F.col("quantity") * F.col("unit_price")) \
    .withColumn("net_amount", F.col("gross_amount") - F.col("discount_amount"))

bad_order_ids_from_items = [r.order_id for r in df_bad_items.select("order_id").distinct().collect()]
valid_order_ids = df_good_items.select("order_id").filter(~F.col("order_id").isin(bad_order_ids_from_items)).distinct()

valid_order_ids.createOrReplaceTempView("v_valid_order_ids")
df_valid.join(valid_order_ids, "order_id", "inner").createOrReplaceTempView("v_valid_orders_final")
df_good_items.filter(F.col("order_id").isin([r.order_id for r in valid_order_ids.collect()])).createOrReplaceTempView("v_valid_items_final")

# Combine quarantine records
df_final_quarantine = df_invalid.select(
    "order_id", F.col("quarantine_reason").alias("reject_reason"), "updated_at", "raw_record"
).unionByName(
    df_bad_items.select("order_id", "reject_reason", "updated_at", "raw_record")
).withColumn("quarantined_at", F.current_timestamp())

df_final_quarantine.createOrReplaceTempView("v_quarantine_incoming")

# 8. C5 - Incremental MERGE INTO
df_valid_orders_final = spark.table("v_valid_orders_final").select(
    "order_id", "customer_id", "store_id", "order_ts", "order_date",
    "order_status", "payment_method", "channel", "promised_mins",
    "actual_mins", "delivery_fee", "updated_at"
)

DeltaTable.forName(spark, f"{catalog}.silver.orders").alias("target").merge(
    df_valid_orders_final.alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdateAll(
    condition="source.updated_at > target.updated_at"
).whenNotMatchedInsertAll().execute()

df_valid_items_final = spark.table("v_valid_items_final").select(
    "order_id", "line_number", "product_id", "quantity", "unit_price",
    "discount_amount", "gross_amount", "net_amount", "updated_at"
)

DeltaTable.forName(spark, f"{catalog}.silver.order_items").alias("target").merge(
    df_valid_items_final.alias("source"),
    "target.order_id = source.order_id AND target.line_number = source.line_number"
).whenMatchedUpdateAll(
    condition="source.updated_at > target.updated_at"
).whenNotMatchedInsertAll().execute()

DeltaTable.forName(spark, f"{catalog}.silver.quarantine_orders").alias("target").merge(
    spark.table("v_quarantine_incoming").alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdate(
    condition="source.updated_at > target.updated_at",
    set={
        "reject_reason": "source.reject_reason",
        "updated_at": "source.updated_at",
        "raw_record": "source.raw_record",
        "quarantined_at": "source.quarantined_at"
    }
).whenNotMatchedInsertAll().execute()